# 02 — Feature Engineering: `bureau` agregasyonu

**Proje:** Credit Risk Scoring (Home Credit Default Risk)

Bu defterde **veri boru hattının ilk gerçek tablo birleştirmesini** yapıyoruz. `bureau.csv` her başvuran (`SK_ID_CURR`) için diğer kredi kuruluşlarındaki **birden fazla** kredi kaydını içerir. `bureau_balance.csv` ise bureau içindeki her kredi (`SK_ID_BUREAU`) için aylık ödeme durumunu tutar.

**Akış**
1. `bureau` ve `bureau_balance` ham tablolarını yükle (memory-efficient).
2. `bureau_balance` → `SK_ID_BUREAU` bazında özetle (status oneH-hot ortalamaları + ay sayısı).
3. `bureau` ile birleştir, sonra `SK_ID_CURR` bazında numerik (min/max/mean/var/sum) ve kategorik (one-hot mean) özetler üret.
4. `application_train` ile `LEFT JOIN`.
5. Memory raporu çıkar ve sonucu `data/processed/application_train_bureau.parquet` olarak kaydet.

> Not: Tüm ağır işler `src/feature_engineering.py`'ye taşındı. Bu defter sadece **orchestration** yapar.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import (
    load_csv,
    missing_value_report,
    reduce_mem_usage,
    PROCESSED_DIR,
)
from src.feature_engineering import (
    aggregate_bureau_balance,
    aggregate_bureau,
    merge_features,
)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print(f"Processed dir: {PROCESSED_DIR}")

## 1) Ham tabloları yükle

`load_csv` zaten downcast yaptığı için belleğin patlamasından korkmuyoruz; ama yine de her tablonun shape ve memory kullanımını gözleyelim.

In [ ]:
app_train = load_csv("application_train.csv")
bureau = load_csv("bureau.csv")
bureau_balance = load_csv("bureau_balance.csv")

print()
print(f"app_train      : {app_train.shape}")
print(f"bureau         : {bureau.shape}")
print(f"bureau_balance : {bureau_balance.shape}")

## 2) `bureau_balance` → `SK_ID_BUREAU` özeti

Her kredi için aylık ödeme durumu (`STATUS`: 0,1,2,3,4,5,C,X) ve `MONTHS_BALANCE` özeti çıkarılır. STATUS sütunları one-hot ortalamasıyla "ayların ne oranında bu durumda olduğu" bilgisine dönüşür.

In [ ]:
bb_agg = aggregate_bureau_balance(bureau_balance)
print(f"bb_agg shape   : {bb_agg.shape}")
print(f"bb_agg columns : {list(bb_agg.columns)}")
bb_agg.head()

## 3) `bureau` → `SK_ID_CURR` özeti

Her başvuran için:
- Numerik kolonlar (DAYS_CREDIT, AMT_CREDIT_SUM, AMT_CREDIT_SUM_DEBT, ...) → min/max/mean/var/sum
- Kategorik kolonlar (CREDIT_ACTIVE, CREDIT_CURRENCY, CREDIT_TYPE) → one-hot oranları
- `CREDIT_ACTIVE = Active/Closed` filtreli ek özetler (BURO_ACTIVE_*, BURO_CLOSED_*)
- `BURO_COUNT`: başvurana ait toplam bureau kayıt sayısı

Bu adım birkaç saniye sürebilir; `bureau` ~1.7M satırlı.

In [ ]:
bureau_agg = aggregate_bureau(bureau, bureau_balance_agg=bb_agg)
print(f"bureau_agg shape : {bureau_agg.shape}")
bureau_agg.head()

## 4) `application_train` ile birleştir

Hedef tabloya `LEFT JOIN`. Eşleşmeyen `SK_ID_CURR` (yani bureau geçmişi olmayan başvuranlar) için tüm yeni sütunlar `NaN` olur — bu, modelin bureau eksikliğini bir sinyal olarak öğrenmesini sağlayan klasik yaklaşım.

In [ ]:
train_with_bureau = merge_features(app_train, bureau_agg, on="SK_ID_CURR")

print(f"app_train          : {app_train.shape}")
print(f"train_with_bureau  : {train_with_bureau.shape}")
print(f"new feature count  : {train_with_bureau.shape[1] - app_train.shape[1]}")

mem_mb = train_with_bureau.memory_usage(deep=True).sum() / 1024**2
print(f"memory usage       : {mem_mb:.2f} MB")

## 5) Hızlı kontrol: en bilgi verici yeni sütunlar

`TARGET` ile **mutlak Pearson korelasyonu** en yüksek 15 yeni sütunu listeleyelim. Bu, eklediğimiz feature'ların gerçekten sinyal taşıyıp taşımadığına dair ilk göstergedir.

In [ ]:
new_cols = [c for c in bureau_agg.columns if c != "SK_ID_CURR"]
numeric_new = [
    c for c in new_cols if pd.api.types.is_numeric_dtype(train_with_bureau[c])
]

corr = (
    train_with_bureau[numeric_new + ["TARGET"]]
    .corr(numeric_only=True)["TARGET"]
    .drop("TARGET")
    .abs()
    .sort_values(ascending=False)
    .head(15)
)
corr.to_frame("abs_corr_with_TARGET")

## 6) Diske kaydet

Aynı agregasyonu her seferinde tekrarlamamak için sonucu `data/processed/application_train_bureau.parquet` olarak kaydediyoruz. Sonraki defter (previous_application + POS_CASH + installments + credit_card) bu dosyayı doğrudan okuyacak.

In [ ]:
out_path = PROCESSED_DIR / "application_train_bureau.parquet"
train_with_bureau.to_parquet(out_path, index=False)
print(f"Saved: {out_path}")
print(f"File size: {out_path.stat().st_size / 1024**2:.2f} MB")